# Install TANGLE and configure Jupyter

TANGLE is not a pure-Python package. Its Python API wraps the Rust
implementation with PyO3, so installation has two parts: Cargo compiles
the Rust workspace, then Maturin installs the resulting extension into
a Python environment. Jupyter must use that same environment as its
kernel.

## Installation plan

1. Install Python 3.10+, stable Rust/Cargo, and a native compiler.
2. Create a virtual environment or named Conda environment and open
   this notebook with that kernel.
3. Install Maturin and Jupyter tooling into the active environment.
4. Let Maturin invoke Cargo and install TANGLE.
5. Register the reusable **Python (TANGLE)** kernel and verify the import.

CubeCL's CPU backend uses LLVM internally, but Cargo downloads CubeCL's
matching LLVM bundle automatically. No separate LLVM installation or
`llvm-config` is required.

## One-time prerequisite: install Rust and Cargo

If `rustc --version` and `cargo --version` already work, skip this
section. Otherwise install Rust with the official Rustup installer.
Cargo is installed alongside Rust.

On macOS, Linux, or Windows Subsystem for Linux:

```bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
source "$HOME/.cargo/env"
rustup toolchain install stable
rustup default stable
rustc --version
cargo --version
```

On Windows, download and run the official
[`rustup-init.exe`](https://rust-lang.org/tools/install/), accept the
stable MSVC toolchain, then open a new PowerShell window:

```powershell
rustup toolchain install stable
rustup default stable
rustc --version
cargo --version
```

Rust also needs a native linker: Xcode command-line tools on macOS,
GCC/Clang development tools on Linux, or Visual Studio Build Tools with
the Desktop C++ workload on Windows. The Windows Rustup installer may
prompt for those build tools. Restart VS Code after installing Rust so
its notebook kernels inherit the updated command path.

### Where Maturin is installed

The setup commands below install Maturin with `pip` inside the active
Python environment. This is intentional: the `maturin` executable then
follows the selected Jupyter kernel and installs TANGLE into that same
environment.

Maturin can instead be installed as a Rust CLI with Cargo:

```bash
cargo install --locked maturin
maturin --version
```

If you choose the Cargo installation, `maturin` is normally placed in
Cargo's executable directory (usually `~/.cargo/bin`). You still need
to activate `.venv-tangle` or the named Conda environment before
running `maturin develop`. The self-running notebook cell deliberately
uses the environment-local `pip` installation, so there is no need to
run both installation methods.

## Manual equivalent: POSIX shell

These are the plain shell commands performed by the setup cell below.
Run them from the TANGLE repository root if you prefer a terminal.

```bash
python3 --version
rustc --version
cargo --version
python3 -m venv .venv-tangle
source .venv-tangle/bin/activate
python -m pip install --upgrade pip
python -m pip install "maturin>=1.8,<2" jupyterlab ipykernel
maturin develop --release --manifest-path crates/tangle_python/Cargo.toml
python -m ipykernel install --user --name tangle --display-name "Python (TANGLE)"
python -c "import tangle; print(tangle.__file__)"
```

## Manual equivalent: Windows PowerShell

```powershell
py -3 --version
rustc --version
cargo --version
py -3 -m venv .venv-tangle
.\.venv-tangle\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install "maturin>=1.8,<2" jupyterlab ipykernel
maturin develop --release --manifest-path crates/tangle_python/Cargo.toml
python -m ipykernel install --user --name tangle --display-name "Python (TANGLE)"
python -c "import tangle; print(tangle.__file__)"
```

These commands assume the one-time Rust/native-compiler prerequisite
above has already been completed.

## Miniforge / Conda alternative

Miniforge can manage the Python environment instead of `venv`. It does
not replace Rust/Cargo or the platform's native compiler. After
[installing Miniforge](https://github.com/conda-forge/miniforge), open
a Conda-enabled terminal and run these commands from the TANGLE
repository root:

```bash
conda create --name tangle python=3.12 -y
conda activate tangle
python -m pip install --upgrade pip
python -m pip install "maturin>=1.8,<2" jupyterlab ipykernel
maturin develop --release --manifest-path crates/tangle_python/Cargo.toml
python -m ipykernel install --user --name tangle --display-name "Python (TANGLE)"
python -c "import tangle; print(tangle.__file__)"
```

Use a named environment rather than installing TANGLE into Conda's
`base` environment. In VS Code or Jupyter, select **Python (TANGLE)**
after registration. If Rust was installed while VS Code was open,
restart VS Code before building so its kernel can find Cargo.

## Run the installation from this notebook

The notebook must already use a virtual-environment or named Conda
kernel; a running kernel cannot replace its own Python interpreter. The
cell finds the repository root, checks Rust/Cargo, installs the Python
build tools, configures Maturin for the active environment, compiles
TANGLE, and registers this exact interpreter as **Python (TANGLE)**.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# A compiled extension must be installed into the same environment as
# this running kernel; installing into system Python will not help it.
active_virtualenv = bool(os.environ.get("VIRTUAL_ENV")) or sys.prefix != sys.base_prefix
active_conda = bool(os.environ.get("CONDA_PREFIX")) or (
    Path(sys.prefix) / "conda-meta"
).is_dir()
if not active_virtualenv and not active_conda:
    raise RuntimeError(
        "This kernel is not using a virtual environment or Conda "
        "environment. Create one with the commands above, then select "
        "its Python kernel."
    )
if active_conda and os.environ.get("CONDA_DEFAULT_ENV") == "base":
    raise RuntimeError(
        "This kernel is using Conda's base environment. Create and "
        "activate the named 'tangle' environment shown above."
    )

# Walk upward so this notebook works from Jupyter or VS Code without
# assuming that either application chose the repository as its CWD.
def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        manifest = candidate / "crates/tangle_python/Cargo.toml"
        if manifest.is_file():
            return candidate
    raise RuntimeError(
        "Could not find crates/tangle_python/Cargo.toml above "
        f"{start}. Open this notebook from the TANGLE checkout."
    )

repository = find_repository_root(Path.cwd().resolve())
print("Repository:", repository)
print("Environment:", sys.prefix)

# Maturin invokes Cargo, so fail early with a useful message when the
# one-time Rust installation is missing from this kernel's PATH.
for executable in ("rustc", "cargo"):
    if shutil.which(executable) is None:
        raise RuntimeError(
            f"{executable} is not available. Install stable Rust with "
            "rustup, restart the shell/VS Code, and try again."
        )

def run(command: list[str], *, environment=None) -> None:
    print("\n$", " ".join(command), flush=True)
    subprocess.run(
        command,
        cwd=repository,
        env=environment,
        check=True,
    )

# Use this interpreter's pip to keep Maturin and the notebook kernel in
# the same environment.
run([
    sys.executable, "-m", "pip", "install", "--upgrade", "--quiet",
    "pip", "maturin>=1.8,<2", "jupyterlab", "ipykernel",
])

# Windows and POSIX environments place console scripts differently.
maturin_name = "maturin.exe" if os.name == "nt" else "maturin"
maturin_candidates = (
    Path(sys.executable).parent / maturin_name,
    Path(sys.prefix) / "Scripts" / maturin_name,
    Path(sys.prefix) / "bin" / maturin_name,
)
maturin = next((path for path in maturin_candidates if path.is_file()), None)
if maturin is None:
    raise RuntimeError(
        f"Maturin was installed but {maturin_name} was not found under {sys.prefix}."
    )

# Remove stale venv/Conda markers before selecting this kernel's Python
# explicitly for PyO3. This prevents builds from landing in another env.
scripts = maturin.parent
build_environment = os.environ.copy()
if active_conda and not os.environ.get("VIRTUAL_ENV"):
    build_environment.pop("VIRTUAL_ENV", None)
    build_environment["CONDA_PREFIX"] = sys.prefix
else:
    build_environment.pop("CONDA_PREFIX", None)
    build_environment.pop("CONDA_DEFAULT_ENV", None)
    build_environment["VIRTUAL_ENV"] = sys.prefix
build_environment["PYO3_PYTHON"] = sys.executable
build_environment["PATH"] = (
    str(scripts) + os.pathsep + build_environment.get("PATH", "")
)

# `develop` creates an editable install, so rebuilding updates this
# checkout without copying Python sources into site-packages.
run([
    str(maturin), "develop", "--release", "--manifest-path",
    "crates/tangle_python/Cargo.toml",
], environment=build_environment)
# Register a stable display name that both Jupyter and VS Code can use.
run([
    sys.executable, "-m", "ipykernel", "install", "--user",
    "--name", "tangle", "--display-name", "Python (TANGLE)",
])

print("\nInstallation complete. Restart this notebook kernel, then run verification.")

## Verify the active kernel

Restart the notebook kernel after the installation cell completes, then
run this cell. Restarting is necessary because Python reads Maturin's
editable-install path file when the kernel starts.

In [ ]:
import importlib.util
import os
import platform
import shutil
import sys
from pathlib import Path

# Print both the interpreter and build tools because the most common
# import failure is a notebook attached to the wrong Python kernel.
print("Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("Platform:", platform.platform())
for executable in ("rustc", "cargo"):
    print(f"{executable:12s}", shutil.which(executable) or "NOT FOUND")
maturin_name = "maturin.exe" if os.name == "nt" else "maturin"
maturin_candidates = (
    Path(sys.executable).parent / maturin_name,
    Path(sys.prefix) / "Scripts" / maturin_name,
    Path(sys.prefix) / "bin" / maturin_name,
)
maturin = shutil.which("maturin") or next(
    (str(path) for path in maturin_candidates if path.is_file()),
    "NOT FOUND",
)
print(f"{'maturin':12s}", maturin)
print(f"{'environment':12s}", os.environ.get("CONDA_DEFAULT_ENV") or sys.prefix)

# `find_spec` checks visibility without importing the native module, so
# the resulting error can still explain how to repair the environment.
if importlib.util.find_spec("tangle") is None:
    raise RuntimeError(
        "TANGLE is not visible. Restart this kernel after installation "
        "and confirm that it uses the same Python environment."
    )
import tangle
print("Loaded:", tangle.__file__)
print("Smoke test:", tangle.Cell([1e-3, 1e-3, 1e-3]).lengths)

## What happened?

- Cargo resolved and compiled the Rust workspace, CubeCL kernels, and
  the automatically downloaded CPU backend dependencies.
- Maturin built the PyO3 extension and installed it as editable package
  `tangle` in the active Python environment.
- IPykernel registered that interpreter for VS Code and JupyterLab.

## Rebuild after Rust binding changes

Re-run the installation cell after changing Rust binding code, then
restart the notebook kernel. Cargo reuses unchanged build artifacts.
Pure notebook or Python-file changes do not require rebuilding.